# Example 4: Retrospective Optimization on Real LNPDB Studies

This notebook demonstrates the two main LNPBO use cases on **real data** from [LNPDB](https://github.com/evancollins1/LNPDB):

1. **Discrete pool BO (retrospective)** — Given a completed screening study, simulate how BO would have found the top formulations using fewer experiments than exhaustive screening.
2. **Continuous ratio BO (prospective)** — For a fixed lipid identity, optimize molar ratios in real-time using a GP surrogate.

**Prerequisites:** LNPDB must be cloned as a sibling directory with symlink at `data/LNPDB_repo` (see README).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from LNPBO.data.lnpdb_bridge import load_lnpdb_full
from LNPBO.data.dataset import Dataset
from LNPBO.optimization.optimizer import Optimizer

## Load LNPDB and select a study

Each LNPDB study is identified by `Publication_PMID` and stratified by `Model_type` (cell line), since z-scores are normalized per (Experiment_ID, Model_type).

In [ ]:
dataset = load_lnpdb_full()
df = dataset.df
print(f"LNPDB: {len(df):,} formulations from {df['Publication_PMID'].nunique()} publications")

In [ ]:
# Select YX_2024 HeLa study (PMID 39060305): 1,200 IL-diverse formulations
study_df = df[
    (df["Publication_PMID"] == 39060305) &
    (df["Model_type"] == "HeLa")
].copy().reset_index(drop=True)

print(f"Study: PMID 39060305 (HeLa)")
print(f"  Formulations: {len(study_df)}")
print(f"  Unique ILs: {study_df['IL_SMILES'].nunique()}")
print(f"  Unique HLs: {study_df['HL_name'].nunique()}")
print(f"  Experiment_value range: [{study_df['Experiment_value'].min():.2f}, {study_df['Experiment_value'].max():.2f}]")

---
## Use Case 1: Discrete Pool BO (Retrospective Simulation)

Simulate a multi-round screening campaign. We start with a random seed pool (25% of formulations), then use BO to select batches of 12 from the remaining pool. After each batch, we reveal the true `Experiment_value` (simulating lab results) and update the surrogate.

**Goal:** Recover the top 5% of formulations using as few experiments as possible.

In [ ]:
# Encode molecular features (LANTERN: count Morgan FP + RDKit descriptors, PCA-reduced)
ds = Dataset(study_df, source="lnpdb", name="YX_2024_HeLa")
encoded = ds.encode_dataset(
    {"IL": {"count_mfp": 5, "rdkit": 5}},
    reduction="pca",
)
enc_df = encoded.df
feature_cols = [c for c in enc_df.columns if c.startswith("IL_count_mfp_pc") or c.startswith("IL_rdkit_pc")]
print(f"Encoded: {len(enc_df)} formulations, {len(feature_cols)} features")
print(f"Features: {feature_cols}")

In [ ]:
# Split into seed pool (25%) and oracle pool (75%)
rng = np.random.RandomState(42)
n_seed = int(0.25 * len(enc_df))
all_idx = np.arange(len(enc_df))
rng.shuffle(all_idx)
seed_idx = sorted(all_idx[:n_seed].tolist())
oracle_idx = sorted(all_idx[n_seed:].tolist())

# Define top-5% threshold
k = max(1, int(0.05 * len(enc_df)))
top_k_values = set(enc_df.nlargest(k, "Experiment_value").index)

print(f"Seed pool: {len(seed_idx)} formulations")
print(f"Oracle pool: {len(oracle_idx)} formulations")
print(f"Top-5% threshold: best {k} formulations")
print(f"Top-5% in seed: {len(top_k_values & set(seed_idx))}/{k}")

In [ ]:
# Run multi-round BO with XGBoost-UCB surrogate
optimizer = Optimizer(
    surrogate="xgb_ucb",
    batch_size=12,
    kappa=5.0,
    normalize="copula",
    random_seed=42,
)

batch_size = 12
n_rounds = 15
training_idx = list(seed_idx)
pool_idx = list(oracle_idx)

# Track recall over rounds
recall_curve = []
found = top_k_values & set(training_idx)
recall_curve.append(len(found) / len(top_k_values))

for r in range(n_rounds):
    if len(pool_idx) < batch_size:
        break

    batch = optimizer.suggest_indices(
        enc_df, feature_cols, training_idx, pool_idx, round_num=r,
    )

    # "Reveal" oracle values (simulate lab testing)
    training_idx.extend(batch)
    pool_idx = [i for i in pool_idx if i not in set(batch)]

    found = top_k_values & set(training_idx)
    recall = len(found) / len(top_k_values)
    recall_curve.append(recall)

    best_val = enc_df.loc[batch, "Experiment_value"].max()
    print(f"Round {r+1:2d}: batch_best={best_val:.2f}, top-5% recall={recall:.1%}")

print(f"\nFinal top-5% recall: {recall_curve[-1]:.1%} (found {len(found)}/{len(top_k_values)})")

In [ ]:
# Plot recall curve
fig, ax = plt.subplots(figsize=(6, 4))
experiments = [n_seed + i * batch_size for i in range(len(recall_curve))]
ax.plot(experiments, recall_curve, "o-", color="#2196F3", label="XGBoost-UCB")
ax.axhline(recall_curve[0], color="gray", ls="--", alpha=0.5, label="Random (seed only)")
ax.set_xlabel("Total experiments")
ax.set_ylabel("Top-5% recall")
ax.set_title("PMID 39060305 (HeLa, N=1200): Retrospective BO Simulation")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

### Compare surrogates

Run the same simulation with different surrogates to see which performs best on this study.

In [ ]:
surrogates = {
    "XGBoost-UCB": {"surrogate": "xgb_ucb"},
    "RF-TS": {"surrogate": "rf_ts"},
    "NGBoost-UCB": {"surrogate": "ngboost"},
}

results = {}
for name, kwargs in surrogates.items():
    opt = Optimizer(batch_size=12, kappa=5.0, normalize="copula", random_seed=42, **kwargs)
    t_idx = list(seed_idx)
    p_idx = list(oracle_idx)
    curve = [len(top_k_values & set(t_idx)) / len(top_k_values)]

    for r in range(n_rounds):
        if len(p_idx) < batch_size:
            break
        batch = opt.suggest_indices(enc_df, feature_cols, t_idx, p_idx, round_num=r)
        t_idx.extend(batch)
        p_idx = [i for i in p_idx if i not in set(batch)]
        curve.append(len(top_k_values & set(t_idx)) / len(top_k_values))

    results[name] = curve
    print(f"{name}: final recall = {curve[-1]:.1%}")

# Plot comparison
fig, ax = plt.subplots(figsize=(6, 4))
colors = {"XGBoost-UCB": "#2196F3", "RF-TS": "#4CAF50", "NGBoost-UCB": "#FF9800"}
for name, curve in results.items():
    exps = [n_seed + i * batch_size for i in range(len(curve))]
    ax.plot(exps, curve, "o-", color=colors[name], label=name, markersize=4)
ax.set_xlabel("Total experiments")
ax.set_ylabel("Top-5% recall")
ax.set_title("Surrogate comparison on PMID 39060305 (HeLa)")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

---
## Use Case 2: Ratio Optimization (Prospective, Continuous BO)

For a study where the ionizable lipid is fixed and only molar ratios vary (e.g., PMID 35879315, HepG2), we use a GP surrogate with continuous acquisition optimization over the ratio simplex.

This simulates a **prospective** campaign: the optimizer suggests new ratio combinations to test.

In [ ]:
# Load a ratio-only study (single IL, varying molar ratios)
ratio_df = df[df["Publication_PMID"] == 35879315].copy().reset_index(drop=True)

print(f"Study: PMID 35879315 (HepG2)")
print(f"  Formulations: {len(ratio_df)}")
print(f"  Unique ILs: {ratio_df['IL_SMILES'].nunique()}")
print(f"  IL: {ratio_df['IL_name'].iloc[0]}")
print(f"  Ratio columns:")
for col in ['IL_molratio', 'HL_molratio', 'CHL_molratio', 'PEG_molratio']:
    if col in ratio_df.columns:
        vals = ratio_df[col].dropna()
        print(f"    {col}: [{vals.min():.2f}, {vals.max():.2f}] ({vals.nunique()} unique)")

In [ ]:
# For ratio-only studies, features are the molar ratios themselves
ds_ratio = Dataset(ratio_df, source="lnpdb", name="YZ_2022_HepG2")
encoded_ratio = ds_ratio.encode_dataset(reduction="none")
enc_ratio_df = encoded_ratio.df

ratio_features = [c for c in enc_ratio_df.columns if "molratio" in c.lower()]
print(f"Ratio features: {ratio_features}")

# Split: small seed (30 formulations), large oracle
rng2 = np.random.RandomState(42)
n_seed_ratio = 30
all_idx2 = np.arange(len(enc_ratio_df))
rng2.shuffle(all_idx2)
seed_idx2 = sorted(all_idx2[:n_seed_ratio].tolist())
oracle_idx2 = sorted(all_idx2[n_seed_ratio:].tolist())

k2 = max(1, int(0.05 * len(enc_ratio_df)))
top_k2 = set(enc_ratio_df.nlargest(k2, "Experiment_value").index)

# Run BO with sklearn GP (continuous acquisition optimization over ratio space)
opt_ratio = Optimizer(
    surrogate="gp_sklearn",
    acquisition_type="UCB",
    kappa=5.0,
    normalize="copula",
    random_seed=42,
    batch_size=12,
)

t_idx = list(seed_idx2)
p_idx = list(oracle_idx2)
curve_ratio = [len(top_k2 & set(t_idx)) / len(top_k2)]

for r in range(10):
    if len(p_idx) < 12:
        break
    batch = opt_ratio.suggest_indices(
        enc_ratio_df, ratio_features, t_idx, p_idx, round_num=r,
    )
    t_idx.extend(batch)
    p_idx = [i for i in p_idx if i not in set(batch)]
    recall = len(top_k2 & set(t_idx)) / len(top_k2)
    curve_ratio.append(recall)
    print(f"Round {r+1:2d}: top-5% recall = {recall:.1%}")

print(f"\nFinal: {curve_ratio[-1]:.1%} recall with {len(t_idx)} experiments (of {len(enc_ratio_df)} total)")

---
## Browse available LNPDB studies

List all studies qualifying for benchmarking (≥200 formulations).

In [ ]:
from LNPBO.benchmarks.benchmark import characterize_studies

studies = characterize_studies(df, min_size=200)
print(f"{len(studies)} qualifying studies:\n")
print(f"{'Study ID':>25s}  {'N':>5s}  {'ILs':>5s}  {'Type':>30s}  {'Cell':>12s}")
print("-" * 85)
for si in studies:
    sid = si.get('study_id', str(int(float(si['pmid']))))
    print(f"{sid:>25s}  {si['n_formulations']:>5d}  {si['n_unique_il']:>5d}  {si['study_type']:>30s}  {si.get('model_type', '?'):>12s}")